# Leksjon 05 - Agentisk RAG


## Oppsett

Denne notatboken demonstrerer Agentic RAG (Retrieval-Augmented Generation) mønsteret ved bruk av Microsoft Agent Framework.

**Forutsetninger:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — din Azure AI Search tjenesteendepunkt
- `AZURE_SEARCH_API_KEY` — din Azure AI Search API-nøkkel
- Azure OpenAI distribusjon konfigurert via miljøvariabler
- Azure CLI autentisert (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hva er Agentic RAG?

Tradisjonell RAG følger en fast pipeline: hente dokumenter, deretter generere et svar. **Agentic RAG** går lenger ved å gi agenten autonomi til å bestemme **når** og **hvordan** informasjon skal hentes.

Med Agentic RAG kan agenten:
- **Bestemme** om henting er nødvendig før den svarer på et spørsmål
- **Velge** hvilken datakilde eller verktøy som skal spørres
- **Evaluere** hentede resultater og gjøre oppfølgingshentinger hvis første forsøk er utilstrekkelig
- **Kombinere** informasjon fra flere hentetrinn til et sammenhengende svar

Dette gjør agenten mer fleksibel og nøyaktig sammenlignet med en statisk hent-deretter-generer pipeline.


## Lage et søkeverktøy

I Agentic RAG blir eksterne datakilder pakket inn som **verktøy** som agenten kan bruke etter behov. Dette lar agenten behandle innhenting som bare en handling den kan utføre, i stedet for et obligatorisk trinn.

Nedenfor definerer vi en kunnskapsdatabase for reise og eksponerer den som et verktøy agenten kan bruke for å slå opp informasjon om reisemål.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Bygge RAG-agenten

Nå lager vi en agent som er instruert til å **alltid hente informasjon før den svarer**. Agenten bruker verktøyet `search_travel_knowledge` for å forankre svarene sine i kunnskapsbasen i stedet for å stole på sin egen treningsdata.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativ henting — Maker-Checker-mønsteret

En viktig fordel med Agentic RAG er **iterativ henting**. Agenten kan utføre flere søkerunder for å verifisere, forbedre eller utvide sine opprinnelige funn — tilsvarende en "maker-checker"-arbeidsflyt:

1. **Maker-steg**: Agenten henter innledende informasjon og utarbeider et svar.
2. **Checker-steg**: Agenten gjør ytterligere søk for å verifisere detaljer eller fylle ut hull.

Nedenfor blir agenten stilt et spørsmål som krever sammenligning av flere destinasjoner, noe som får den til å søke flere ganger.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Sammendrag

I denne leksjonen lærte du hvordan du bygger et **Agentic RAG**-system ved hjelp av Microsoft Agent Framework:

- **Agentic RAG** lar agenter selvstendig bestemme når de skal hente informasjon, noe som gjør innhentingen dynamisk i stedet for fast.
- **Verktøy som datakilder**: Eksterne kunnskapsdatabaser (som Azure AI Search) pakkes inn som verktøy agenten kan bruke.
- **Iterativ innhenting**: Maker-checker-mønsteret gjør det mulig for agenten å utføre flere innhentingsrunder — søke, verifisere og forbedre — før den leverer et endelig svar.

I produksjon ville du erstatte den midlertidige `TRAVEL_KNOWLEDGE_BASE`-minnelagringen med en ekte Azure AI Search-indeks for å håndtere innhenting av reisedokumenter i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på dets opprinnelige språk skal anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
